# Interactive Complex Number Demonstrations

This notebook contains two minimal interactive visualisations:

1. **Plotly** — rotating complex number and Euler's formula on the unit circle
2. **ipywidgets** — slider-controlled domain colouring parameters

Both examples are designed to build geometric intuition without heavy computation.

Install the package first if you haven't already:
```bash
pip install -e '.[dev]'
```

## 1. Euler's Formula — Plotly animation

Euler's formula states:
$$e^{i\theta} = \cos\theta + i\sin\theta$$

As \(\theta\) sweeps from \(0\) to \(2\pi\), the point \(e^{i\theta}\) traces the unit circle.
Below is a **static** Plotly figure showing the unit circle with several marked angles.

In [ ]:
import numpy as np
import plotly.graph_objects as go

theta = np.linspace(0, 2 * np.pi, 300)
circle_x = np.cos(theta)
circle_y = np.sin(theta)

# Marked angles: multiples of π/4
angles = np.arange(0, 2 * np.pi, np.pi / 4)
pts_x = np.cos(angles)
pts_y = np.sin(angles)
labels = [f"e^(i·{k}π/4)" for k in range(len(angles))]

fig = go.Figure()

# Unit circle
fig.add_trace(go.Scatter(
    x=circle_x, y=circle_y,
    mode='lines',
    line=dict(color='steelblue', width=2),
    name='Unit circle'
))

# Radii to marked angles
for px, py in zip(pts_x, pts_y):
    fig.add_trace(go.Scatter(
        x=[0, px], y=[0, py],
        mode='lines',
        line=dict(color='gray', width=1, dash='dot'),
        showlegend=False
    ))

# Marked points
fig.add_trace(go.Scatter(
    x=pts_x, y=pts_y,
    mode='markers+text',
    marker=dict(size=10, color='crimson'),
    text=labels,
    textposition='top right',
    name='e^(iθ)'
))

# Origin
fig.add_trace(go.Scatter(
    x=[0], y=[0],
    mode='markers',
    marker=dict(size=8, color='black'),
    name='Origin'
))

fig.update_layout(
    title='Euler\'s Formula: e^(iθ) traces the unit circle',
    xaxis=dict(title='Re', range=[-1.5, 1.5], zeroline=True, zerolinewidth=1),
    yaxis=dict(title='Im', range=[-1.5, 1.5], scaleanchor='x', zeroline=True, zerolinewidth=1),
    width=600, height=600,
    showlegend=True,
    template='plotly_white'
)

fig.show()

## 2. Animated rotation — complex multiplication

Multiplying by \(e^{i\phi}\) **rotates** a complex number \(z\) by angle \(\phi\).

The animation below shows the point \(z = 1.2 + 0.5i\) being rotated around the origin.

In [ ]:
import numpy as np
import plotly.graph_objects as go

z0 = 1.2 + 0.5j
theta_frames = np.linspace(0, 2 * np.pi, 60)
circle_t = np.linspace(0, 2 * np.pi, 200)

# Build animation frames
frames = []
for th in theta_frames:
    z = z0 * np.exp(1j * th)
    frames.append(go.Frame(
        data=[
            go.Scatter(x=[0, z.real], y=[0, z.imag],
                       mode='lines+markers',
                       line=dict(color='crimson', width=3),
                       marker=dict(size=[6, 12], color='crimson')),
        ]
    ))

# Initial state
z_init = z0
fig = go.Figure(
    data=[
        go.Scatter(x=np.cos(circle_t) * abs(z0),
                   y=np.sin(circle_t) * abs(z0),
                   mode='lines', line=dict(color='steelblue', dash='dot'),
                   name=f'|z₀| = {abs(z0):.2f}'),
        go.Scatter(x=[0, z_init.real], y=[0, z_init.imag],
                   mode='lines+markers',
                   line=dict(color='crimson', width=3),
                   marker=dict(size=[6, 12], color='crimson'),
                   name='z₀·e^(iθ)'),
    ],
    frames=frames,
    layout=go.Layout(
        title='Rotation: z₀·e^(iθ)',
        xaxis=dict(range=[-2, 2], zeroline=True),
        yaxis=dict(range=[-2, 2], scaleanchor='x', zeroline=True),
        width=600, height=600,
        template='plotly_white',
        updatemenus=[dict(
            type='buttons',
            showactive=False,
            buttons=[
                dict(label='▶ Play',
                     method='animate',
                     args=[None, dict(frame=dict(duration=50, redraw=True),
                                      fromcurrent=True)]),
                dict(label='⏹ Pause',
                     method='animate',
                     args=[[None], dict(frame=dict(duration=0, redraw=False),
                                        mode='immediate')]),
            ]
        )]
    )
)

fig.show()

## 3. ipywidgets — interactive domain colouring parameters

Use the slider to change the **brightness parameter** \(a\) in the domain-colouring map.
Smaller \(a\) → brighter colours near poles; larger \(a\) → more contrast near zeros.

In [ ]:
%matplotlib widget
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from cnlecture.cplotting import colorize

n = 250
x = np.linspace(-2, 2, n)
y = np.linspace(-2, 2, n)
X, Y = np.meshgrid(x, y)
Z = X + 1j * Y

# f(z) = (z^2 - 1) / (z^2 + 1)  — four branch points visible
f = (Z**2 - 1) / (Z**2 + 1)

fig, ax = plt.subplots(figsize=(6, 6))
lim = 2

img_plot = ax.imshow(
    colorize(f, a=0.5),
    extent=[-lim, lim, -lim, lim],
    origin='upper',
    interpolation='none'
)
ax.set_xlabel('Re')
ax.set_ylabel('Im')
ax.set_title(r'Domain colouring of $(z^2-1)/(z^2+1)$')
fig.tight_layout()


@widgets.interact(a=widgets.FloatSlider(value=0.5, min=0.01, max=0.99, step=0.01,
                                        description='brightness a'))
def update(a):
    img_plot.set_data(colorize(f, a=a))
    fig.canvas.draw_idle()